# DKI legacy: batched vs. single per-sample integration

Runs the **original** DKI training recipe (`legacy/baseline_runner.py`) two ways on the
*same data, seed and weights*, and compares them:

* **single** — the original loop: each sample integrated on its own `odeint` call
  (`for i in range(batch_size)`), using the non-batch-safe `ODEFunc`.
* **batched** (`--batched`) — the whole minibatch integrated in one `odeint` call over a
  `(B, N)` state, using the batch-safe `ODEFuncBatched`.

Both share the same two-`Linear` fitness and start from identical weights under a fixed
seed, so they produce **identical** loss curves — the comparison is purely about
*how* integration is dispatched and what that costs in wall-clock.

> **Note on speed:** batching mainly pays off on **GPU**. On CPU, with the original's
> dense time grid `t = arange(0, 100, 0.01)` (10,000 output points), the two paths do the
> same total work and run at roughly the same speed.

## 1. Setup

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/metagenAu/DKI.git'
BRANCH   = 'claude/loving-knuth-Ju4Xr'   # change to 'main' once merged
REPO_DIR = '/content/DKI'

if not os.path.exists(REPO_DIR):
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO_DIR])
%cd $REPO_DIR
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

import torch, numpy as np
from dki.device import auto_device
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, 'device', DEVICE)

## 2. Pick your data

**Bundled** — set `USE_BUNDLED = True` to use the gLV synthetic data shipped with the repo.

**Your own** — keep `USE_BUNDLED = False` and upload an abundance CSV (and optionally a
separate test CSV). The flags below describe your file layout; the cell rewrites it into the
`(n_taxa, n_samples)` orientation the DKI loader expects.

In [ ]:
USE_BUNDLED = False             # set False to upload your own
samples_as_rows = True          # only matters when USE_BUNDLED=False
header_row = True             # set True if your CSV has a header row
index_col  = True             # set True if your CSV has a row-label column

DATA_DIR = '/content/DKI/data' if USE_BUNDLED else '/content/dki_data'

if not USE_BUNDLED:
    from google.colab import files
    os.makedirs(DATA_DIR, exist_ok=True)
    print('Upload your abundance CSV (and optionally a test CSV).')
    uploaded = files.upload()
    import pandas as pd
    for name, _ in uploaded.items():
        df = pd.read_csv(name,
                         header=0 if header_row else None,
                         index_col=0 if index_col else None)
        arr = df.to_numpy(dtype=np.float32)
        if samples_as_rows:
            arr = arr.T    # -> (n_taxa, n_samples) for the DKI loader
        dst = os.path.join(DATA_DIR, 'Ptrain.csv' if 'train' in name.lower() or len(uploaded)==1
                                       else 'Ptest.csv')
        np.savetxt(dst, arr, delimiter=',')
        print(f'  wrote {dst}  shape={arr.shape}  (taxa, samples)')

print('Data dir:', DATA_DIR)
!ls -la $DATA_DIR

## 3. Run both paths

Same `--data`, `--epochs`, `--batch-size` and `--seed` for both, so the only difference is
`--batched`. Each run writes `val_loss.npy` and `epoch_times.npy` to its own output folder.

Keep `EPOCHS` small to start — the original dense-grid solve makes each epoch heavy
(seconds to minutes per epoch on CPU, depending on the number of taxa).

In [ ]:
EPOCHS     = 5
BATCH_SIZE = 20
SEED       = 0

OUT_SINGLE  = '/content/results/legacy_single'
OUT_BATCHED = '/content/results/legacy_batched'

def run_legacy(out_dir, batched):
    cmd = [sys.executable, 'legacy/baseline_runner.py',
           '--data', DATA_DIR, '--epochs', str(EPOCHS),
           '--batch-size', str(BATCH_SIZE), '--seed', str(SEED),
           '--device', DEVICE, '--out', out_dir]
    if batched:
        cmd.append('--batched')
    print('>', ' '.join(cmd))
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(proc.stdout[-2000:])
    if proc.returncode != 0:
        print(proc.stderr[-2000:])
        raise RuntimeError('legacy run failed')

run_legacy(OUT_SINGLE,  batched=False)
run_legacy(OUT_BATCHED, batched=True)

## 4. Compare

Load each run's saved arrays. The validation curves should be **identical** (same dynamics,
same weights); the wall-clock is what differs.

In [ ]:
import pandas as pd

val_single  = np.load(os.path.join(OUT_SINGLE,  'val_loss.npy'))
val_batched = np.load(os.path.join(OUT_BATCHED, 'val_loss.npy'))
t_single    = np.load(os.path.join(OUT_SINGLE,  'epoch_times.npy'))
t_batched   = np.load(os.path.join(OUT_BATCHED, 'epoch_times.npy'))

summary = pd.DataFrame([
    {'path': 'single',  'best_val_BC': val_single.min(),
     'mean_sec/epoch': t_single.mean(),  'total_sec': t_single.sum()},
    {'path': 'batched', 'best_val_BC': val_batched.min(),
     'mean_sec/epoch': t_batched.mean(), 'total_sec': t_batched.sum()},
])
speedup = t_single.mean() / t_batched.mean()
print(f'batched speedup (single/batched epoch time): {speedup:.2f}x  on {DEVICE}')

max_curve_diff = np.abs(val_single - val_batched).max()
print(f'max |val_BC difference| between paths: {max_curve_diff:.2e}  (expected ~0)')
summary

## 5. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

epochs = np.arange(1, len(val_single) + 1)
ax1.plot(epochs, val_single,  'o-', label='single',  lw=2)
ax1.plot(epochs, val_batched, 'x--', label='batched', lw=2)
ax1.set_xlabel('epoch'); ax1.set_ylabel('validation Bray-Curtis')
ax1.set_title('Val BC (curves overlap = identical dynamics)')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.bar(['single', 'batched'], [t_single.mean(), t_batched.mean()],
        color=['#4c72b0', '#dd8452'])
ax2.set_ylabel('mean seconds / epoch')
ax2.set_title(f'Wall-clock per epoch on {DEVICE}  ({speedup:.2f}x)')
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout(); plt.show()

## Takeaways

* The two paths give the **same** validation curve and best val BC — `--batched` changes
  only how the integration is dispatched, not the model or its dynamics.
* On **GPU** the batched path should be markedly faster (one vectorised `(B, N)` solve vs.
  `B` separate `(1, N)` solves). On **CPU** with the original dense time grid the two are
  roughly even, because total function evaluations are the same and small problems don't
  vectorise much.
* For the bigger speedup that the refactored `dki/` package gets, batching is combined with
  a 2-point time span (`t=[0, t_final]`) instead of 10,000 output points — see
  `notebooks/dki_colab.ipynb`.